<a href="https://colab.research.google.com/github/makenzieearle/Modeling-Probability-of-the-Schrodinger-Equation/blob/main/ElectronScatteringSimulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- IMPORTS ---
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve
from IPython.display import HTML


hbar = 1.054571817e-34       # Planck constant (J·s)
m = 9.10938356e-31           # electron mass (kg)

L = 1e-8                     # length of 1D region (m)
N = 1000                     # number of spatial points
dx = L / N                   # spatial step
dt = 1e-18                   # time step (s)
steps = 600                  # total time steps
x = np.linspace(0, L, N)     # spatial grid

# --- INITIAL WAVE PACKET (electron) ---
x0 = L / 4                   # initial center position
sigma = L / 40               # width of Gaussian
k = 7e10                     # wavenumber (controls energy)
psi = np.exp(-(x - x0)**2 / (4 * sigma**2)) * np.exp(1j * k * x)
psi /= np.sqrt(np.sum(np.abs(psi)**2) * dx)  # normalization
psi[0] = psi[-1] = 0         # Dirichlet boundaries

# --- POTENTIAL BARRIER ---
V0 = 1e-17                   # height of barrier (J)
V = np.zeros(N)
V[N//2 - 50 : N//2 + 50] = V0  # square barrier in middle

# --- CRANK–NICOLSON MATRICES ---
main_diag = -2.0 * np.ones(N)
off_diag = np.ones(N - 1)
laplacian = diags([off_diag, main_diag, off_diag], [-1, 0, 1]) / dx**2
H = -(hbar**2 / (2*m)) * laplacian + diags(V, 0)
I = diags([np.ones(N)], [0])
A = I + 1j * dt / (2*hbar) * H
B = I - 1j * dt / (2*hbar) * H

fig, ax = plt.subplots(figsize=(7, 3))
ax.set_xlim(0, L * 1e9)
ax.set_ylim(0, 1.2 * np.max(np.abs(psi)**2))
ax.set_xlabel("Position (nm)")
ax.set_ylabel("Probability Density |ψ|²")
ax.set_title("Electron Scattering (Crank–Nicolson Simulation)")

line_prob, = ax.plot([], [], 'k-', lw=1.5, label='|ψ|²')
line_V, = ax.plot(x * 1e9, V / V0 * np.max(np.abs(psi)**2), 'r--', label='Potential Barrier')
ax.legend()


def init():
    line_prob.set_data([], [])
    return line_prob, line_V


def animate(frame):
    global psi
    psi = spsolve(A, B @ psi)
    psi[0] = psi[-1] = 0
    psi /= np.sqrt(np.sum(np.abs(psi)**2) * dx)
    line_prob.set_data(x * 1e9, np.abs(psi)**2)
    return line_prob, line_V

ani = FuncAnimation(fig, animate, frames=steps, init_func=init,
                    blit=True, interval=20, repeat=False)

HTML(ani.to_jshtml())


/tmp/ipython-input-1998234265.py:61: SparseEfficiencyWarning: spsolve requires A be CSC or CSR matrix format
  psi = spsolve(A, B @ psi)
